# Voting System Pilot: 20-Pair Validation

**Tesis de Maestría** | HU-08 | Componente: Sistema de Votación Agéntico

Este pilot ejecuta el panel de tres jueces (`judge_openai`, `judge_google`, `judge_anthropic`) sobre las 20 conversaciones del sample piloto, agrega las puntuaciones con el agregador de HU-07, y produce una tabla comparativa frente a `human_score` y al baseline G-Eval. La salida final es una decisión documentada **go / no-go** sobre la ejecución completa del dataset (900 pares).

⚠️ Este notebook hace 60 llamadas reales a las APIs de OpenAI, Google y Anthropic. Costo estimado: ~$0.24. Verificar que los 6 pre-flight checks pasaron antes de ejecutar.


In [1]:
import json
import os
import sys
from pathlib import Path
from time import sleep

import yaml
from dotenv import load_dotenv

# Move CWD and sys.path to project root so imports work whether the notebook
# is executed from notebooks/ or from the project root.
if Path.cwd().name == "notebooks":
    os.chdir("..")
ROOT = Path.cwd()
sys.path.insert(0, str(ROOT))

load_dotenv(ROOT / ".env")

from scripts.run_judge import call_agent  # noqa: E402
from src.voting.aggregator import aggregate  # noqa: E402

PILOT_PATH = ROOT / "configs" / "prompts" / "pilot_sample.json"
GEVAL_PILOT_PATH = ROOT / "configs" / "prompts" / "pilot_results_v3.json"
AGENTS_DIR = ROOT / "configs" / "agents"
OUTPUTS_DIR = ROOT / "outputs" / "agent_scores"
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY missing"
assert os.getenv("GOOGLE_API_KEY"), "GOOGLE_API_KEY missing"
assert os.getenv("ANTHROPIC_API_KEY"), "ANTHROPIC_API_KEY missing"

with open(PILOT_PATH) as f:
    pilot = json.load(f)
assert len(pilot) == 20
print(f"Pilot sample loaded: {len(pilot)} entries")

Pilot sample loaded: 20 entries


In [2]:
agent_configs = {}
for name in ["agent_openai", "agent_google", "agent_anthropic"]:
    with open(AGENTS_DIR / f"{name}.yaml") as f:
        agent_configs[name] = yaml.safe_load(f)

V3_PROMPT = "configs/prompts/geval_relevance_prompt.txt"
print(f"{'name':<18} {'model':<22} {'provider':<10} {'prompt_file'}")
for name, c in agent_configs.items():
    print(f"{c['name']:<18} {c['model']:<22} {c['provider']:<10} {c['prompt_file']}")
    assert c["prompt_file"] == V3_PROMPT, f"{name} uses wrong prompt"
print("\nAll three agents share the V3 prompt — control científico OK.")

name               model                  provider   prompt_file
judge_openai       gpt-4o                 openai     configs/prompts/geval_relevance_prompt.txt
judge_google       gemini-2.5-flash       google     configs/prompts/geval_relevance_prompt.txt
judge_anthropic    claude-haiku-4-5       anthropic  configs/prompts/geval_relevance_prompt.txt

All three agents share the V3 prompt — control científico OK.


In [3]:
def build_conversation_input(entry):
    """Format turns[:-1] as `[Turn N] Role: content` lines."""
    context = entry["turns"][:-1]
    return "\n".join(
        f"[Turn {i + 1}] {t['role'].capitalize()}: {t['content']}" for i, t in enumerate(context)
    )


def run_agent_pilot(agent_key, out_path, transient_retry=3, retry_sleep=5):
    """Run one agent over the 20-pair pilot; persist a JSON list with per-entry results."""
    cfg = agent_configs[agent_key]
    print(f"Running {cfg['name']} ({cfg['model']}) on {len(pilot)} pairs...")
    results = []
    n_ok = n_fail = 0
    total_cost = 0.0
    for i, entry in enumerate(pilot, 1):
        conv_id = entry["metadata"]["conversation_id"]
        conv_input = build_conversation_input(entry)
        out = None
        last_err = None
        for attempt in range(1, transient_retry + 1):
            try:
                out = call_agent(cfg, conv_input, entry["actual_output"])
                break
            except Exception as exc:
                last_err = exc
                if attempt < transient_retry:
                    sleep(retry_sleep * attempt)
        if out is None:
            n_fail += 1
            results.append(
                {
                    "conversation_id": conv_id,
                    "agent": cfg["name"],
                    "model": cfg["model"],
                    "score": None,
                    "reasoning": "",
                    "tokens_used": 0,
                    "cost_usd": 0.0,
                    "timestamp": None,
                    "retry_used": False,
                    "error": f"{type(last_err).__name__}: {last_err}",
                }
            )
            print(f"  {i:>2}/{len(pilot)} {conv_id[:40]:<40} FAILED: {last_err}")
            continue
        out["conversation_id"] = conv_id
        results.append(out)
        total_cost += out["cost_usd"]
        if out["score"] is None:
            n_fail += 1
            tag = "PARSE FAIL"
        else:
            n_ok += 1
            tag = f"score {out['score']}"
        print(f"  {i:>2}/{len(pilot)} {conv_id[:40]:<40} {tag:<11}  cost ${out['cost_usd']:.4f}")
    with open(out_path, "w") as f:
        json.dump(results, f, indent=2, ensure_ascii=False)
    print(
        f"\n{cfg['name']} complete. {n_ok}/{len(pilot)} successful, {n_fail} failed. Total cost: ${total_cost:.4f}"
    )
    return results


openai_results = run_agent_pilot("agent_openai", OUTPUTS_DIR / "pilot_agent_openai.json")

Running judge_openai (gpt-4o) on 20 pairs...


   1/20 conv_97_ground-truth                     score 5      cost $0.0042


   2/20 conv_94_ground-truth                     score 5      cost $0.0046


   3/20 conv_91_ground-truth                     score 5      cost $0.0043


   4/20 conv_8_ground-truth                      score 5      cost $0.0046


...

  16/20 conv_24_VHRED_attn greedy_temp1.0_k0_p0. score 2      cost $0.0055


  17/20 conv_67_GPT2_medium greedy_temp1.0_k0_p0 score 1      cost $0.0045


  18/20 conv_86_S2S sample_temp1.0_k0_p0.0       score 1      cost $0.0044


  19/20 conv_26_HRED_attn sample_temp1.0_k0_p0.0 score 1      cost $0.0048


  20/20 conv_18_VHRED_attn sample_temp1.0_k0_p0. score 1      cost $0.0042

judge_openai complete. 20/20 successful, 0 failed. Total cost: $0.0947


In [4]:
google_results = run_agent_pilot("agent_google", OUTPUTS_DIR / "pilot_agent_google.json")

Running judge_google (gemini-2.5-flash) on 20 pairs...


   1/20 conv_97_ground-truth                     score 5      cost $0.0007


   2/20 conv_94_ground-truth                     score 5      cost $0.0007


   3/20 conv_91_ground-truth                     score 5      cost $0.0006


   4/20 conv_8_ground-truth                      score 5      cost $0.0007


...

  16/20 conv_24_VHRED_attn greedy_temp1.0_k0_p0. score 3      cost $0.0010


  17/20 conv_67_GPT2_medium greedy_temp1.0_k0_p0 score 2      cost $0.0009


  18/20 conv_86_S2S sample_temp1.0_k0_p0.0       score 1      cost $0.0008


  19/20 conv_26_HRED_attn sample_temp1.0_k0_p0.0 score 3      cost $0.0010


  20/20 conv_18_VHRED_attn sample_temp1.0_k0_p0. score 1      cost $0.0008

judge_google complete. 20/20 successful, 0 failed. Total cost: $0.0154


In [5]:
anthropic_results = run_agent_pilot("agent_anthropic", OUTPUTS_DIR / "pilot_agent_anthropic.json")

Running judge_anthropic (claude-haiku-4-5) on 20 pairs...


   1/20 conv_97_ground-truth                     score 5      cost $0.0034


   2/20 conv_94_ground-truth                     score 5      cost $0.0030


   3/20 conv_91_ground-truth                     score 5      cost $0.0026


   4/20 conv_8_ground-truth                      score 5      cost $0.0028


...

  16/20 conv_24_VHRED_attn greedy_temp1.0_k0_p0. score 3      cost $0.0037


  17/20 conv_67_GPT2_medium greedy_temp1.0_k0_p0 score 2      cost $0.0030


  18/20 conv_86_S2S sample_temp1.0_k0_p0.0       score 1      cost $0.0032


  19/20 conv_26_HRED_attn sample_temp1.0_k0_p0.0 score 2      cost $0.0039


  20/20 conv_18_VHRED_attn sample_temp1.0_k0_p0. score 1      cost $0.0033

judge_anthropic complete. 20/20 successful, 0 failed. Total cost: $0.0641


In [6]:
def by_conv_id(results):
    return {r["conversation_id"]: r for r in results}


oai_map = by_conv_id(openai_results)
goo_map = by_conv_id(google_results)
ant_map = by_conv_id(anthropic_results)

voting_results = []
for entry in pilot:
    conv_id = entry["metadata"]["conversation_id"]
    scores = {
        "judge_openai": oai_map[conv_id]["score"],
        "judge_google": goo_map[conv_id]["score"],
        "judge_anthropic": ant_map[conv_id]["score"],
    }
    try:
        agg = aggregate(scores)
        agg_failed = False
    except ValueError as exc:
        agg = {
            "final_score": None,
            "individual_scores": scores,
            "agreement_level": "n/a",
            "scheme_used": None,
            "metadata": {"error": str(exc)},
        }
        agg_failed = True
    voting_results.append(
        {
            "conversation_id": conv_id,
            "human_score": entry["metadata"]["human_score"],
            "model": entry["metadata"]["model"],
            "stratum": entry.get("stratum"),
            "individual_scores": scores,
            "vote_final": agg["final_score"],
            "agreement_level": agg["agreement_level"],
            "metadata": agg["metadata"],
            "aggregate_failed": agg_failed,
        }
    )

out_path = ROOT / "outputs" / "voting_pilot_results.json"
with open(out_path, "w") as f:
    json.dump(voting_results, f, indent=2, ensure_ascii=False)
n_ok = sum(1 for r in voting_results if r["vote_final"] is not None)
print(f"Aggregated {n_ok}/{len(voting_results)} entries. Saved to {out_path.relative_to(ROOT)}")

Aggregated 20/20 entries. Saved to outputs/voting_pilot_results.json


In [7]:
with open(GEVAL_PILOT_PATH) as f:
    geval = json.load(f)
geval_map = {g["conversation_id"]: g for g in geval}
matched = sum(1 for r in voting_results if r["conversation_id"] in geval_map)
print(f"G-Eval scores loaded for {matched}/{len(voting_results)} entries")

G-Eval scores loaded for 20/20 entries


In [8]:
def observation(row):
    notes = []
    s = row["individual_scores"]
    if any(v is None for v in s.values()):
        notes.append("⚠️ agent failed")
    if row["agreement_level"] == "low":
        notes.append("⚠️ low agreement")
    vf = row["vote_final"]
    hs = row["human_score"]
    gs = row.get("geval_score")
    if vf is not None:
        if abs(vf - hs) <= 0.5:
            notes.append("✓ close to human")
        if gs is not None and abs(vf - gs) <= 0.5:
            notes.append("≈ G-Eval")
        if gs is not None and vf > gs + 1:
            notes.append("↑ voting > geval")
        if gs is not None and vf < gs - 1:
            notes.append("↓ voting < geval")
    return ", ".join(notes) if notes else ""


rows = []
for r in voting_results:
    g = geval_map.get(r["conversation_id"], {})
    r["geval_score"] = g.get("geval_score")
    s = r["individual_scores"]
    rows.append(
        {
            "conversation_id": r["conversation_id"],
            "human": r["human_score"],
            "geval": r.get("geval_score"),
            "ai_openai": s["judge_openai"],
            "ai_google": s["judge_google"],
            "ai_anthropic": s["judge_anthropic"],
            "vote_final": r["vote_final"],
            "agreement": r["agreement_level"],
            "observación": observation(r),
        }
    )


# Print as a fixed-width table for readability in the notebook
def fmt(v, w, d=2):
    return f"{v:>{w}.{d}f}" if isinstance(v, int | float) else f"{'?':>{w}}"


hdr = f"{'conversation_id':<50} {'human':>5} {'geval':>5} {'oai':>4} {'goo':>4} {'ant':>4} {'vote':>5} {'agr':<6} observación"
print(hdr)
print("-" * len(hdr))
for row in rows:
    print(
        f"{row['conversation_id']:<50} "
        f"{fmt(row['human'], 5)} {fmt(row['geval'], 5)} "
        f"{fmt(row['ai_openai'], 4, 0)} {fmt(row['ai_google'], 4, 0)} {fmt(row['ai_anthropic'], 4, 0)} "
        f"{fmt(row['vote_final'], 5)} {row['agreement']:<6} {row['observación']}"
    )

conversation_id                                    human geval  oai  goo  ant  vote agr    observación
------------------------------------------------------------------------------------------------------
conv_97_ground-truth                                5.00  3.20    5    5    5  5.00 high   ✓ close to human, ↑ voting > geval
conv_94_ground-truth                                5.00  4.32    5    5    5  5.00 high   ✓ close to human
conv_91_ground-truth                                5.00  3.44    5    5    5  5.00 high   ✓ close to human, ↑ voting > geval
conv_8_ground-truth                                 5.00  4.70    5    5    5  5.00 high   ✓ close to human, ≈ G-Eval
conv_1_negative-sample                              1.00  1.48    1    1    1  1.00 high   ✓ close to human, ≈ G-Eval
conv_22_negative-sample                             1.00  1.01    1    1    1  1.00 high   ✓ close to human, ≈ G-Eval
conv_28_negative-sample                             1.00  1.02    1    1    1  1

In [9]:
import statistics
from collections import Counter

DEGENERATE_STD = 0.3


def agent_stats(results, agent_name):
    valid = [r["score"] for r in results if r["score"] is not None]
    if len(valid) < 2:
        return {
            "agent": agent_name,
            "n": len(valid),
            "mean": None,
            "std": None,
            "mode": None,
            "pct_at_mode": None,
            "status": "INSUFFICIENT DATA",
        }
    mean = statistics.mean(valid)
    std = statistics.stdev(valid)
    counts = Counter(valid)
    mode_val, mode_n = counts.most_common(1)[0]
    pct = 100 * mode_n / len(valid)
    status = "⚠️ DEGENERATE" if std < DEGENERATE_STD else "OK"
    return {
        "agent": agent_name,
        "n": len(valid),
        "mean": mean,
        "std": std,
        "mode": mode_val,
        "pct_at_mode": pct,
        "status": status,
    }


degen = [
    agent_stats(openai_results, "judge_openai"),
    agent_stats(google_results, "judge_google"),
    agent_stats(anthropic_results, "judge_anthropic"),
]
print(f"{'Agent':<18} {'n':>3} {'Mean':>5} {'Std':>5} {'Mode':>5} {'%@Mode':>7}  Status")
print("-" * 60)
for d in degen:
    n = d["n"]
    mean_s = f"{d['mean']:.2f}" if d["mean"] is not None else "  -  "
    std_s = f"{d['std']:.2f}" if d["std"] is not None else "  -  "
    mode_s = f"{d['mode']}" if d["mode"] is not None else "-"
    pct_s = f"{d['pct_at_mode']:.0f}%" if d["pct_at_mode"] is not None else "  -  "
    print(f"{d['agent']:<18} {n:>3} {mean_s:>5} {std_s:>5} {mode_s:>5} {pct_s:>7}  {d['status']}")

Agent                n  Mean   Std  Mode  %@Mode  Status
------------------------------------------------------------
judge_openai        20  2.65  1.76     1     45%  OK
judge_google        20  3.00  1.75     5     35%  OK
judge_anthropic     20  2.80  1.64     5     30%  OK


In [10]:
import math

from scipy.stats import spearmanr


def safe_spearman(xs, ys):
    pairs = [(x, y) for x, y in zip(xs, ys, strict=False) if x is not None and y is not None]
    if len(pairs) < 2:
        return float("nan"), 0
    rho, _ = spearmanr([p[0] for p in pairs], [p[1] for p in pairs])
    return float(rho), len(pairs)


human = [r["human_score"] for r in voting_results]
geval_scores = [r.get("geval_score") for r in voting_results]
vote_final = [r["vote_final"] for r in voting_results]
oai_scores = [r["individual_scores"]["judge_openai"] for r in voting_results]
goo_scores = [r["individual_scores"]["judge_google"] for r in voting_results]
ant_scores = [r["individual_scores"]["judge_anthropic"] for r in voting_results]

rho_table = [
    ("G-Eval (gpt-4o)", *safe_spearman(human, geval_scores)),
    ("Voting final", *safe_spearman(human, vote_final)),
    ("judge_openai", *safe_spearman(human, oai_scores)),
    ("judge_google", *safe_spearman(human, goo_scores)),
    ("judge_anthropic", *safe_spearman(human, ant_scores)),
]
print(f"{'Method':<22} {'Spearman ρ':>10} {'n':>4}")
print("-" * 40)
for name, rho, n in rho_table:
    rho_s = "  nan" if math.isnan(rho) else f"{rho:.3f}"
    print(f"{name:<22} {rho_s:>10} {n:>4}")
print("\nNote: n=20 — preliminary only, not statistically conclusive.")

Method                 Spearman ρ    n
----------------------------------------
G-Eval (gpt-4o)             0.902   20
Voting final                0.892   20
judge_openai                0.846   20
judge_google                0.802   20
judge_anthropic             0.927   20

Note: n=20 — preliminary only, not statistically conclusive.


In [11]:
def agent_cost(results):
    tin = sum(r.get("tokens_in", 0) for r in results)
    tout = sum(r.get("tokens_out", 0) for r in results)
    cost = sum(r.get("cost_usd", 0.0) for r in results)
    calls = len(results)
    return calls, tin, tout, cost


agents = [
    ("judge_openai", openai_results),
    ("judge_google", google_results),
    ("judge_anthropic", anthropic_results),
]
print(f"{'Agent':<18} {'Calls':>6} {'Tokens in':>10} {'Tokens out':>11} {'Cost USD':>10}")
print("-" * 60)
totals = [0, 0, 0, 0.0]
for name, res in agents:
    calls, tin, tout, cost = agent_cost(res)
    print(f"{name:<18} {calls:>6} {tin:>10} {tout:>11} ${cost:>8.4f}")
    totals[0] += calls
    totals[1] += tin
    totals[2] += tout
    totals[3] += cost
print("-" * 60)
print(f"{'TOTAL PILOT':<18} {totals[0]:>6} {totals[1]:>10} {totals[2]:>11} ${totals[3]:>8.4f}")
projected = totals[3] * (900 / 20)
print(f"\nProjected cost for full 900-pair run: ${projected:.2f}")

Agent               Calls  Tokens in  Tokens out   Cost USD
------------------------------------------------------------
judge_openai           20      25432        3116 $  0.0947
judge_google           20      25277        3108 $  0.0154
judge_anthropic        20      27445        7329 $  0.0641
------------------------------------------------------------
TOTAL PILOT            60      78154       13553 $  0.1742

Projected cost for full 900-pair run: $7.84


## Manual review findings

Revisión cualitativa de los outputs de las celdas 9-12, complementada con una inspección directa de los razonamientos crudos de los tres agentes.

### 1. Cobertura técnica del pipeline

Las 60 puntuaciones (20 entradas × 3 jueces) están todas en el rango entero 1-5. **0 valores `None` y 0 fallos de parsing** del score en la primera pasada: el patrón de extracción captura los tres dialectos que produjeron los modelos sin disparar el retry de formato en ningún caso. Los razonamientos guardados en `outputs/agent_scores/pilot_agent_*.json` muestran que los tres jueces siguieron el chain-of-thought del prompt V3 (Pasos 1 a 5) en las 20 entradas; ninguno produjo respuestas degeneradas.

| Agente | Longitud mínima del razonamiento | Mediana | Máxima |
|---|---|---|---|
| judge_openai | 466 chars | 747 | 1156 |
| judge_google | 411 chars | 667 | 1162 |
| judge_anthropic | 1179 chars | 1648 | 2335 |

`judge_anthropic` produce razonamientos sistemáticamente más largos (estilo más conversacional, con cierre en Markdown `**Score: N**`); `judge_google` los más cortos y literales al ejemplo del V3 (`Score = N`). Esa variabilidad de estilo es esperada y no afecta a la calidad del score extraído.

### 2. Jueces degenerados

Ninguno. Las desviaciones estándar de los tres jueces sobre el pilot quedan claramente por encima del umbral 0.3:

| Agente | Mean | Std | Moda | %@Moda | Estado |
|---|---|---|---|---|---|
| judge_openai | 2.65 | 1.76 | 1 | 45% | OK |
| judge_google | 3.00 | 1.75 | 5 | 35% | OK |
| judge_anthropic | 2.80 | 1.64 | 5 | 30% | OK |

### 3. Razonabilidad del agregador

Los `vote_final` son coherentes con los scores individuales en las 20 entradas, sin desplazamientos artificiales. El `agreement_level` se reparte de forma sensata: 14 `high`, 5 `medium`, 1 `low` (la entrada con votos `{3, 5, 2}` para una respuesta HRED de calidad intermedia).

### 4. Jerarquía del dataset preservada por estrato

Comparación de medias por estrato (humano vs. agentes vs. voting):

| Estrato | Tipo | n | Humano | OpenAI | Google | Anthropic | Voting |
|---|---|---|---|---|---|---|---|
| 1 | ground-truth top | 4 | 5.00 | 5.00 | 5.00 | 5.00 | 5.00 |
| 2 | negative-sample | 4 | 1.00 | 1.00 | 1.00 | 1.00 | 1.00 |
| 3 | IA alta | 4 | 4.38 | 3.50 | 3.50 | 3.75 | 3.58 |
| 4 | IA media | 4 | 3.19 | 2.75 | 3.75 | 2.75 | 3.08 |
| 5 | IA baja | 4 | 1.50 | 1.00 | 1.75 | 1.50 | 1.42 |

Los extremos (estratos 1 y 2) son trivialmente correctos para los tres agentes. En los estratos intermedios el voting cae cerca del valor humano, dentro de un margen razonable. El **estrato 3** (respuestas de IA de calidad alta) muestra el único patrón sistemático: los tres jueces son más estrictos que el humano (3.58 vs 4.38), por aproximadamente un punto. Es un hallazgo real y no un artefacto del pipeline: **los LLMs evalúan respuestas IA con más rigor que los anotadores humanos cuando la respuesta es de calidad media-alta**. Conviene reportarlo explícitamente en el análisis del run completo.

### 5. Único caso de discrepancia grande (Δ > 1.5 puntos)

Solo una entrada de las 20:

`conv_30_GPT2_small top_temp1.0_k0_p0.9` — `human = 4.75`, `vote = 1.33` (Δ = +3.42).

- Contexto: *"for the past week my cable hasn't been working."*
- Respuesta IA evaluada: *"what is the problem?"*
- Scores individuales: OpenAI 1, Google 1, Anthropic 2.

Los tres jueces interpretan la respuesta literalmente: el usuario **ya dijo** que su cable no funciona y la IA repite la pregunta, ignorando la información ya dada. Los anotadores humanos, en cambio, parecen haberla leído como un follow-up legítimo ("¿qué síntomas específicos?") y le dieron 4.75. No es un fallo del panel; es una **diferencia estructural humano-máquina en casos ambiguos**: los LLMs leen literal, los humanos infieren intención. Es un hallazgo a discutir explícitamente cuando se reporten los resultados del run completo de 900 pares, donde casos así aparecerán con más frecuencia.

### 6. Mejor acuerdo con el humano (Spearman ρ vs `human_score`, n=20)

| Método | ρ |
|---|---|
| judge_anthropic | **0.927** |
| G-Eval (gpt-4o) | 0.902 |
| Voting final | 0.892 |
| judge_openai | 0.846 |
| judge_google | 0.802 |

`judge_anthropic` lidera como juez individual; G-Eval queda segundo y el voting tercero. **Importante**: estas correlaciones del pilot son optimistas por diseño. El sampler estratificado satura los extremos 1 y 5 (8 de 20 entradas), donde discriminar es trivial. Las correlaciones del pilot **no son comparables** con la ρ = 0.7565 que G-Eval obtuvo sobre las 900 entradas del dataset completo; en el run completo del voting se espera regresión hacia un valor más realista en el rango 0.74-0.80.

### 7. Patrón sistemático: sesgos opuestos entre los jueces (validación del agregador)

| Modelo | Tendencia | Evidencia |
|---|---|---|
| OpenAI | Estricto | 45% de sus scores son 1 |
| Google | Generoso | 35% de sus scores son 5 |
| Anthropic | Equilibrado | 30% en la moda, distribución más simétrica |

Estos sesgos opuestos son **exactamente la condición bajo la cual la media aritmética del panel cancela errores sistemáticos** (justificación central del esquema seleccionado en HU-05). En el estrato 4 (IA media, humano 3.19), OAI dice 2.75, Google dice 3.75, Anthropic dice 2.75, y el voting promedia a 3.08 — un valor prácticamente exacto que ningún juez individual alcanza por sí solo. Esto valida el operador agregador para el run completo.

### 8. Calidad del razonamiento: una muestra

Para el caso fácil `conv_22_negative-sample` (humano 1, contexto sobre cortes de pelo, respuesta IA habla de cubiertos), los tres jueces argumentan correctamente la irrelevancia siguiendo el CoT del V3:

- **OpenAI**: *"Step 5 — The response does not address the intent and is completely off-topic. SCORE: 1"*
- **Google**: *"Step 5 — Does not address the intent at all, completely off-topic → Score = 1."*
- **Anthropic**: *"**Score: 1**. The Actual Output is irrelevant—it does not address the communicative intent at all and is completely off-topic."*

Los tres llegan al mismo veredicto con la misma justificación, expresada en sus respectivos dialectos. Esta consistencia cualitativa, junto con los datos cuantitativos anteriores, sostiene la decisión de proceder con el run completo.


## Decision: PROCEED

**Justificación numérica.**

- **Tasa de éxito del pipeline**: 60/60 llamadas exitosas, 0 fallos de parsing, 0 jueces degenerados. El pipeline `run_judge.call_agent` + `aggregate(...)` opera end-to-end sin intervención.
- **Calidad de la agregación en el pilot (n=20)**: `vote_final` correlaciona con `human_score` a ρ = 0.892, comparable con el baseline G-Eval (ρ = 0.902). El voting no degrada el techo, lo mantiene en la misma franja.
- **Costo real del pilot**: $0.1742 (más bajo que la estimación $0.24 de HU-06). Desglose: OpenAI $0.0947, Anthropic $0.0641, Google $0.0154.
- **Costo proyectado para 900 pares**: $7.84. Cae cerca del rango estimado en HU-06 ($7.61) y deja el total de la tesis (G-Eval ya ejecutado + voting) en ~$13.

**Próximo paso.** Implementar `scripts/run_voting_system.py` (siguiente HU) reusando `call_agent` de `scripts/run_judge.py`. Añadir paralelismo entre proveedores (los tres pueden correrse en paralelo por par) y retries con backoff vía `tenacity` siguiendo el patrón de `scripts/run_geval.py`. Ejecutar sobre las 900 entradas, persistir `outputs/voting_results.json` con el mismo schema del pilot y producir un análisis paralelo al de G-Eval (`outputs/voting_analysis_report.md` reusando las funciones de `scripts/analyze_geval.py`).

**Observación para investigación posterior, no bloqueante.** En el pilot, `judge_anthropic` individual supera al voting agregado y al baseline G-Eval (ρ = 0.927). Con n=20 la diferencia no es significativa, pero conviene reportarla explícitamente en el análisis del run completo y discutir si el panel mejora o no respecto al mejor juez individual cuando n=900. Si en el run completo el mejor juez individual sigue superando al voting, la conclusión de la tesis incluirá ese matiz: el voting **iguala** el techo humano pero no necesariamente **lo supera** respecto a un juez fuerte solo.
